In [ ]:
import os
from dotenv import load_dotenv
print(load_dotenv())


True


In [5]:
API_KEY = os.getenv("API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = API_KEY

llm = init_chat_model(
    "llama-3.3-70b-versatile", 
    model_provider="groq",
    temperature=0
)

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("contract1.pdf")
loader2 = PyPDFLoader("contract2.pdf")

docs1 = loader1.load()
docs2 = loader2.load()

# Add source tagging
for doc in docs1:
    doc.metadata["source"] = "Contract A"

for doc in docs2:
    doc.metadata["source"] = "Contract B"

documents = docs1 + docs2

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

In [10]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Example
texts = ["This is contract A clause", "This is contract B clause"]

embeddings = model.encode(texts)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\RESHMI\OneDrive\Desktop\Contract\Contract_clause_team_project\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RESHMI\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer

class CustomEmbeddings:
    def __init__(self):
        self.model = model
    
    def embed_documents(self, texts):
        return self.model.encode(texts).tolist()
    
    def embed_query(self, text):
        return self.model.encode([text])[0].tolist()

In [ ]:
from langchain_community.vectorstores import FAISS

embeddings = CustomEmbeddings()

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [ ]:
query = "Compare risks between Contract A and Contract B"

In [ ]:
relevant_docs = retriever.invoke(query)

In [ ]:
context = "\n\n".join([
    f"{doc.metadata.get('source', 'Unknown')}:\n{doc.page_content}"
    for doc in relevant_docs
])

In [ ]:
prompt = f"""
You are a legal expert.

Compare Contract A and Contract B based on the context.

Identify:
- Risks
- Conflicts
- Liability issues
- Which contract is riskier and why

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response)